<a href="https://colab.research.google.com/github/Malujoro/SI_classificacao/blob/main/Codigo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalação de dependências

In [ ]:
!pip install ucimlrepo

---
# Importações
---

In [ ]:
import numpy as np

from IPython.display import display
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import cohen_kappa_score, get_scorer_names, make_scorer, get_scorer
from sklearn.pipeline import Pipeline

from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB

---
# Parâmetros globais
---

In [ ]:
label = "Grade"
quantFolds = 5
porcentTeste = 0.3

kappa_scorer = make_scorer(cohen_kappa_score)

scoring = {
  'acuracia': 'accuracy',
  'precisao': 'precision',
  'recall': 'recall',
  'f1_score': 'f1',
  'kappa': kappa_scorer
}

---
# Funções Auxiliares
---

## Classe Metricas

In [ ]:
class Metricas:

  @staticmethod
  def separarTupla(valor):
    if(isinstance(valor, tuple)):
      media, desvio = valor
      return f"{float(media):.4f} ± {float(desvio):.4f}"
    return  f"{float(valor):.4f}"

  @staticmethod
  def exibirMetricas(titulo, dados, copy=True):
    dados_formatados = {}
    for chave, valor in dados.items():
      dados_formatados[chave] = Metricas.separarTupla(valor)

    print("=" * 40)
    print(titulo)
    print("-" * 40)
    print(f"Acurácia: {dados_formatados['acuracia']}")
    print(f"Precisão: {dados_formatados['precisao']}")
    print(f"Recall: {dados_formatados['recall']}")
    print(f"F1 Score: {dados_formatados['f1_score']}")
    print(f"Kappa: {dados_formatados['kappa']}")
    print("=" * 40)
    if(copy):
      print("Área para copiar os dados")
      for item in dados_formatados.values():
        print(item)

  def __init__(self):
    self.acuracia = []
    self.recall = []
    self.precisao = []
    self.f1_score = []
    self.kappa = []

  def atualizarMetricas(self, dicio):
    self.acuracia.extend(dicio["test_acuracia"])
    self.precisao.extend(dicio["test_precisao"])
    self.recall.extend(dicio["test_recall"])
    self.f1_score.extend(dicio["test_f1_score"])
    self.kappa.extend(dicio["test_kappa"])

  def getMedias(self):
    return {
        "acuracia": np.mean(self.acuracia),
        "precisao": np.mean(self.precisao),
        "recall": np.mean(self.recall),
        "f1_score": np.mean(self.f1_score),
        "kappa": np.mean(self.kappa),
    }

  def getDesvios(self):
    return {
        "acuracia": np.std(self.acuracia),
        "precisao": np.std(self.precisao),
        "recall": np.std(self.recall),
        "f1_score": np.std(self.f1_score),
        "kappa": np.std(self.kappa),
    }

  def exibirTudo(self, text, medias=None, desvios=None, copy=True):
    if(medias == None):
      medias = self.getMedias()

    if(desvios == None):
      desvios = self.getDesvios()

    tuplas = {}
    for chave, valor in medias.items():
      tuplas[chave] = (valor, desvios[chave])

    Metricas.exibirMetricas(text, tuplas)

## Função treinar_modelo

In [ ]:
def treinar_modelo(pipeline):
  kfold = StratifiedKFold(n_splits=quantFolds, shuffle=True, random_state=42)

  scores = cross_validate(pipeline, X_train, y_train, cv=kfold, scoring=scoring)

  metricas = Metricas()
  metricas.atualizarMetricas(scores)

  metricas.exibirTudo("Cross Validation (média ± desvio)")

  pipeline.fit(X_train, y_train)

  return pipeline

## Função testar_modelo

In [ ]:
def testar_modelo(pipeline):
  resultados = {}

  y_pred = pipeline.predict(X_test)

  for nome, scorer in scoring.items():
    if(isinstance(scorer, str)):
      scorer_obj = get_scorer(scorer)
    else:
      scorer_obj = scorer

    func = scorer_obj._score_func

    result = func(y_test, y_pred)
    resultados[nome] = result

  Metricas.exibirMetricas(f"Conjunto de teste ({(porcentTeste * 100):.1f} %)", resultados)

## Função executar_modelo

In [ ]:
def executar_modelo(modelo, params=None):
  pipeline = gerar_pipeline(modelo)

  if(params is not None):
    params_fixado = prefixar(params)
    melhor_modelo = executar_grid(pipeline, params_fixado)
  else:
    melhor_modelo = pipeline

  pipeline_treinado = treinar_modelo(melhor_modelo)
  testar_modelo(pipeline_treinado)

## Função gerar_pipeline

In [ ]:
def gerar_pipeline(modelo):
  colunas_numericas = X_train.select_dtypes(include=np.number).columns
  colunas_categoricas = X_train.select_dtypes(exclude=np.number).columns

  if (isinstance(modelo, MultinomialNB)):
    normalizador = MinMaxScaler()
  else:
    normalizador = StandardScaler()

  preprocessor = ColumnTransformer(
    transformers=[
      ("num", normalizador, colunas_numericas),
      ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), colunas_categoricas)
    ]
  )

  return Pipeline([
      ("preprocessamento", preprocessor),
      ("modelo", modelo)
  ])

## Função prefixar

In [ ]:
def prefixar(params, prefixo="modelo__"):
  dicio = {}
  for chave, valor in params.items():
    dicio[f"{prefixo}{chave}"] = valor
  return dicio

## Função executar_grid

In [ ]:
def executar_grid(pipeline, params):
  searcher = GridSearchCV(
      estimator=pipeline,
      param_grid=params,
      cv=quantFolds,
      n_jobs=-1,
      verbose=3,
  )

  searcher.fit(X_train, y_train)

  print("~" * 40)
  print(f"Melhor Score: {searcher.best_score_}")
  print(f"Melhores hiperparâmetros encontrados: {searcher.best_params_}")

  return searcher.best_estimator_

---
# Implementação
---


## Leitura da base

### Leitura

In [ ]:
base_glioma = fetch_ucirepo(id=759)

X = base_glioma.data.features
y = base_glioma.data.targets

In [ ]:
base_glioma.variables

,name,role,type,demographic,description,units,missing_values
0,Grade,Target,Categorical,None,"Glioma grade class information (0 = ""LGG""; 1 =...",N/A,no
1,Gender,Feature,Categorical,Gender,"Gender (0 = ""male""; 1 = ""female"")",N/A,no
2,Age_at_diagnosis,Feature,Continuous,Age,Age at diagnosis with the calculated number of...,years,no
3,Race,Feature,Categorical,Race,"Race (0 = ""white""; 1 = ""black or african Ame...",N/A,no
4,IDH1,Feature,Categorical,None,isocitrate dehydrogenase (NADP(+))1 (0 = NOT_M...,N/A,no
5,TP53,Feature,Categorical,None,tumor protein p53 (0 = NOT_MUTATED; 1 = MUTATED),N/A,no
6,ATRX,Feature,Categorical,None,ATRX chromatin remodeler (0 = NOT_MUTATED; 1 =...,N/A,no
7,PTEN,Feature,Categorical,None,phosphatase and tensin homolog (0 = NOT_MUTATE...,N/A,no
8,EGFR,Feature,Categorical,None,epidermal growth factor receptor (0 = NOT_MUTA...,N/A,no
9,CIC,Feature,Categorical,None,capicua transcriptional repressor (0 = NOT_MUT...,N/A,no


In [ ]:
X.head()

,Gender,Age_at_diagnosis,Race,IDH1,TP53,ATRX,PTEN,EGFR,CIC,MUC16,...,FUBP1,RB1,NOTCH1,BCOR,CSMD3,SMARCA4,GRIN2A,IDH2,FAT4,PDGFRA
0,0,51.30,white,1,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,0,38.72,white,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,0,35.17,white,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,32.78,white,1,1,1,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
4,0,31.51,white,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
y.head()

,Grade
0,0
1,0
2,0
3,0
4,0


### Dados da base lida

In [ ]:
X.shape

(839, 23)

In [ ]:
X.describe()

,Gender,Age_at_diagnosis,IDH1,TP53,ATRX,PTEN,EGFR,CIC,MUC16,PIK3CA,...,FUBP1,RB1,NOTCH1,BCOR,CSMD3,SMARCA4,GRIN2A,IDH2,FAT4,PDGFRA
count,839.000000,839.000000,839.000000,839.000000,839.000000,839.000000,839.000000,839.000000,839.000000,839.000000,...,839.000000,839.000000,839.000000,839.000000,839.000000,839.000000,839.000000,839.000000,839.000000,839.000000
mean,0.418355,50.935411,0.481526,0.414779,0.258641,0.168057,0.133492,0.132300,0.116806,0.087008,...,0.053635,0.047676,0.045292,0.034565,0.032181,0.032181,0.032181,0.027414,0.027414,0.026222
std,0.493583,15.702339,0.499957,0.492978,0.438149,0.374140,0.340309,0.339019,0.321380,0.282015,...,0.225431,0.213206,0.208068,0.182784,0.176586,0.176586,0.176586,0.163383,0.163383,0.159889
min,0.000000,14.420000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,38.055000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,51.550000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,62.800000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,89.290000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Valores nulos / ausentes

In [ ]:
X.isnull().sum()

,0
Gender,0
Age_at_diagnosis,0
Race,0
IDH1,0
TP53,0
ATRX,0
PTEN,0
EGFR,0
CIC,0
MUC16,0


### Valores duplicados

In [ ]:
X.duplicated().sum()

np.int64(1)

### Proporção de rótulos

In [ ]:
display(y[label].value_counts()),
print()
display(y[label].value_counts(normalize=True))

,count
Grade,
0,487
1,352


,proportion
Grade,
0,0.580453
1,0.419547


## Divisão treinamento e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=porcentTeste, stratify=y, random_state=42
)

# Evitar erro por estar "(tamanho, 1)", transformando em "(tamanho, )"
y_train = y_train[label]
y_test = y_test[label]

y_train.value_counts().to_dict()

{0: 341, 1: 246}

---
# Execuções
---

## KNN

In [ ]:
parametros_knn = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"],
}

knn = KNeighborsClassifier()
executar_modelo(knn, parametros_knn)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Melhor Score: 0.8689700130378096
Melhores hiperparâmetros encontrados: {'modelo__metric': 'manhattan', 'modelo__n_neighbors': 11, 'modelo__weights': 'uniform'}
Cross Validation (média ± desvio)
----------------------------------------
Acurácia: 0.8723 ± 0.0277
Precisão: 0.8122 ± 0.0583
Recall: 0.9147 ± 0.0453
F1 Score: 0.8578 ± 0.0259
Kappa: 0.7431 ± 0.0526
Área para copiar os dados
0.8723 ± 0.0277
0.8122 ± 0.0583
0.9147 ± 0.0453
0.8578 ± 0.0259
0.7431 ± 0.0526
Conjunto de teste (30.0 %)
----------------------------------------
Acurácia: 0.8452
Precisão: 0.7863
Recall: 0.8679
F1 Score: 0.8251
Kappa: 0.6869
Área para copiar os dados
0.8452
0.7863
0.8679
0.8251
0.6869


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


## GaussianNB

In [ ]:
parametros_gaussian = {
    "var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5],
}

gaussian = GaussianNB()
executar_modelo(gaussian, parametros_gaussian)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Melhor Score: 0.8381428364479213
Melhores hiperparâmetros encontrados: {'modelo__var_smoothing': 1e-05}
Cross Validation (média ± desvio)
----------------------------------------
Acurácia: 0.8162 ± 0.0866
Precisão: 0.7348 ± 0.1071
Recall: 0.9268 ± 0.0401
F1 Score: 0.8143 ± 0.0685
Kappa: 0.6418 ± 0.1563
Área para copiar os dados
0.8162 ± 0.0866
0.7348 ± 0.1071
0.9268 ± 0.0401
0.8143 ± 0.0685
0.6418 ± 0.1563
Conjunto de teste (30.0 %)
----------------------------------------
Acurácia: 0.8452
Precisão: 0.7557
Recall: 0.9340
F1 Score: 0.8354
Kappa: 0.6924
Área para copiar os dados
0.8452
0.7557
0.9340
0.8354
0.6924


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


## BernoulliNB

In [ ]:
parametros_bernoulli = {
    "alpha": [1e-9, 1e-3, 1e-1],
    "binarize": [None, 0.0],
    "fit_prior": [True, False],
}

bernoulli = BernoulliNB()
executar_modelo(bernoulli, parametros_bernoulli)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Melhor Score: 0.8688975807619876
Melhores hiperparâmetros encontrados: {'modelo__alpha': 1e-09, 'modelo__binarize': 0.0, 'modelo__fit_prior': False}
Cross Validation (média ± desvio)
----------------------------------------
Acurácia: 0.8655 ± 0.0359
Precisão: 0.8048 ± 0.0521
Recall: 0.9024 ± 0.0474
F1 Score: 0.8493 ± 0.0371
Kappa: 0.7289 ± 0.0703
Área para copiar os dados
0.8655 ± 0.0359
0.8048 ± 0.0521
0.9024 ± 0.0474
0.8493 ± 0.0371
0.7289 ± 0.0703
Conjunto de teste (30.0 %)
----------------------------------------
Acurácia: 0.8611
Precisão: 0.7983
Recall: 0.8962
F1 Score: 0.8444
Kappa: 0.7198
Área para copiar os dados
0.8611
0.7983
0.8962
0.8444
0.7198


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


## MultinomialNB

In [ ]:
parametros_multinomial = {
    "alpha": [1e-3, 1e-2, 1e-1],
    "fit_prior": [True, False],
}

multinomial = MultinomialNB()
executar_modelo(multinomial, parametros_multinomial)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Melhor Score: 0.8688830943068231
Melhores hiperparâmetros encontrados: {'modelo__alpha': 0.001, 'modelo__fit_prior': False}
Cross Validation (média ± desvio)
----------------------------------------
Acurácia: 0.8655 ± 0.0351
Precisão: 0.8036 ± 0.0552
Recall: 0.9064 ± 0.0542
F1 Score: 0.8498 ± 0.0364
Kappa: 0.7292 ± 0.0686
Área para copiar os dados
0.8655 ± 0.0351
0.8036 ± 0.0552
0.9064 ± 0.0542
0.8498 ± 0.0364
0.7292 ± 0.0686
Conjunto de teste (30.0 %)
----------------------------------------
Acurácia: 0.8611
Precisão: 0.7983
Recall: 0.8962
F1 Score: 0.8444
Kappa: 0.7198
Área para copiar os dados
0.8611
0.7983
0.8962
0.8444
0.7198


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


## Multilayer Perceptron

In [ ]:
parametros_mlp = {
    "hidden_layer_sizes": [(32,), (64,), (32,16), (64,32)],
    "activation": ["relu", "tanh"],
    "solver": ["adam", "lbfgs"],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate": ["constant", "adaptive"],
}

mlp = MLPClassifier(random_state=42, early_stopping=True)
executar_modelo(mlp, parametros_mlp)

Fitting 5 folds for each of 96 candidates, totalling 480 fits
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Melhor Score: 0.8620165145588874
Melhores hiperparâmetros encontrados: {'modelo__activation': 'relu', 'modelo__alpha': 0.0001, 'modelo__hidden_layer_sizes': (64,), 'modelo__learning_rate': 'constant', 'modelo__solver': 'adam'}
Cross Validation (média ± desvio)
----------------------------------------
Acurácia: 0.8485 ± 0.0295
Precisão: 0.8002 ± 0.0459
Recall: 0.8582 ± 0.0712
F1 Score: 0.8254 ± 0.0353
Kappa: 0.6922 ± 0.0598
Área para copiar os dados
0.8485 ± 0.0295
0.8002 ± 0.0459
0.8582 ± 0.0712
0.8254 ± 0.0353
0.6922 ± 0.0598
Conjunto de teste (30.0 %)
----------------------------------------
Acurácia: 0.8492
Precisão: 0.7931
Recall: 0.8679
F1 Score: 0.8288
Kappa: 0.6946
Área para copiar os dados
0.8492
0.7931
0.8679
0.8288
0.6946


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
